In [0]:
from pyspark.sql import SparkSession, functions as F
import logging

# 1. SETUP & CONFIGURATION
catalogName = "workspace"
schemaName = "capstone_project"
targetSchema = f"{catalogName}.{schemaName}"

# Initialize logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("SilverLayer")

# Ensure the destination schema is available
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {targetSchema}")

class SilverTransformer:
    def __init__(self, spark, schemaPath):
        self.spark = spark
        self.schemaPath = schemaPath

    def fetchTable(self, tableName):
        """Load a table from the bronze layer."""
        return self.spark.read.table(f"{self.schemaPath}.bronze_{tableName}")

    def inspectDataQuality(self, dataFrame, tableName):
        """Audit the dataframe for null values."""
        print(f"\n--- Data Quality Audit: {tableName} ---")
        nullCounts = dataFrame.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in dataFrame.columns])
        nullCounts.show()

    def cleanAndStandardize(self, factDf, regionDf, tableName):
        """Handle orphan records, standardize labels, and preserve Year."""
        initialCount = factDf.count()
        
        # Determining the correct name for the rural/urban column
        areaCol = "Total__Rural__Urban" if "Total__Rural__Urban" in factDf.columns else "Total_Rural_Urban"
        
        # Standardize categorical columns to Title Case
        categoryCols = ["Gender", "Age_Group", "Ethnic_Group", areaCol, 
                        "Literacy_Status", "Employment_Status", "Sector"]
        
        for col in categoryCols:
            if col in factDf.columns:
                factDf = factDf.withColumn(col, F.initcap(F.trim(F.col(col))))

        # Join with region master to validate geography and remove orphan records
        # This preserves all columns from factDf (including 'Year')
        cleanedDf = factDf.join(
            regionDf.select("State_Code", "District_Code", "State_Name", "District_Name"),
            ["State_Code", "District_Code"], "inner"
        ).filter(F.col("Value").isNotNull())

        print(f"--- Transformation Stats: {tableName} ---")
        print(f"Rows removed (Inconsistent/Null): {initialCount - cleanedDf.count()}")
        return cleanedDf

    def createAggregations(self, popDf, litDf, empDf):
        """Generating 5 unique analytical metrics, separating data by Year."""
        
        areaCol = "Total__Rural__Urban" if "Total__Rural__Urban" in popDf.columns else "Total_Rural_Urban"
        
        # 1. Yearly Population by Gender
        popDf.groupBy("State_Name", "Year", "Gender").agg(F.sum("Value").alias("Total_Pop")) \
            .write.format("delta").mode("overwrite").saveAsTable(f"{self.schemaPath}.viz_yearly_gender")
        
        # 2. Yearly Literacy Trends
        litDf.filter(F.col("Literacy_Status") == "Literate") \
            .groupBy("State_Name", "Year", areaCol).agg(F.sum("Value").alias("Literate_Count")) \
            .write.format("delta").mode("overwrite").saveAsTable(f"{self.schemaPath}.viz_yearly_literacy")
        
        # 3. Yearly Employment Sector Distribution
        empDf.filter(F.col("Sector") != "Not Applicable") \
            .groupBy("State_Name", "Year", "Sector").agg(F.sum("Value").alias("Sector_Total")) \
            .write.format("delta").mode("overwrite").saveAsTable(f"{self.schemaPath}.viz_yearly_employment")
        
        # 4. Yearly Geographic Split (Rural/Urban)
        popDf.groupBy("State_Name", "Year", areaCol).agg(F.sum("Value").alias("Area_Total")) \
            .write.format("delta").mode("overwrite").saveAsTable(f"{self.schemaPath}.viz_yearly_geography")
        
        # 5. Yearly Age Group Demographics
        popDf.groupBy("Year", "Age_Group").agg(F.sum("Value").alias("Age_Total")) \
            .write.format("delta").mode("overwrite").saveAsTable(f"{self.schemaPath}.viz_yearly_age_groups")
            
        print("\n[SUCCESS] All 5 Yearly Aggregate Tables created.")

# 2. EXECUTION PHASE
try:
    silverManager = SilverTransformer(spark, targetSchema)
    masterRegion = silverManager.fetchTable("region")
    processedTables = {}

    # Core Cleaning Loop
    for name in ["population", "literacy", "employment"]:
        raw = silverManager.fetchTable(name)
        silverManager.inspectDataQuality(raw, name)
        
        cleaned = silverManager.cleanAndStandardize(raw, masterRegion, name)
        processedTables[name] = cleaned
        cleaned.write.format("delta").mode("overwrite").saveAsTable(f"{targetSchema}.silver_{name}_final")

    # Trigger Aggregations
    silverManager.createAggregations(
        popDf=processedTables["population"],
        litDf=processedTables["literacy"],
        empDf=processedTables["employment"]
    )

    print("\n" + "="*50)
    print("SILVER LAYER PIPELINE COMPLETED SUCCESSFULLY")
    print("="*50)

except Exception as err:
    print(f"Error during Silver Layer processing: {err}")

# 3. VERIFICATION & PREVIEWS
print("\n--- Displaying Previews for New Yearly Aggregate Metrics ---")
vizTables = ["viz_yearly_gender", "viz_yearly_literacy", "viz_yearly_employment", "viz_yearly_geography", "viz_yearly_age_groups"]

for tableName in vizTables:
    print(f"\nPreview for table: {tableName}")
    try:
        spark.read.table(f"{targetSchema}.{tableName}").show(5, truncate=False)
    except Exception as e:
        print(f"Could not display {tableName}: {e}")


--- Data Quality Audit: population ---
+----------+-------------+---------------------------------------+----+----+---------+------+------------+-------------------+-----+-------------------+
|State_Code|District_Code|India__State__Union_Territory__District|Name|Year|Age_Group|Gender|Ethnic_Group|Total__Rural__Urban|Value|ingestion_timestamp|
+----------+-------------+---------------------------------------+----+----+---------+------+------------+-------------------+-----+-------------------+
|         0|            0|                                      0|   0|   0|        0|     0|           0|                  0|    0|                  0|
+----------+-------------+---------------------------------------+----+----+---------+------+------------+-------------------+-----+-------------------+

--- Transformation Stats: population ---
Rows removed (Inconsistent/Null): 5670

--- Data Quality Audit: literacy ---
+----------+-------------+---------------------------------------+----+----+

In [0]:

class SilverTransformer:
    def __init__(self, spark, schemaPath):
        self.spark = spark
        self.schemaPath = schemaPath

    def fetchTable(self, tableName):
        return self.spark.read.table(f"{self.schemaPath}.bronze_{tableName}")

    def cleanAndStandardize(self, factDf, regionDf, tableName):
        areaCol = "Total__Rural__Urban" if "Total__Rural__Urban" in factDf.columns else "Total_Rural_Urban"
        categoryCols = ["Gender", "Age_Group", "Ethnic_Group", areaCol, "Literacy_Status", "Employment_Status", "Sector"]
        
        for col in categoryCols:
            if col in factDf.columns:
                factDf = factDf.withColumn(col, F.initcap(F.trim(F.col(col))))

        return factDf.join(
            regionDf.select("State_Code", "District_Code", "State_Name", "District_Name"),
            ["State_Code", "District_Code"], "inner"
        ).filter(F.col("Value").isNotNull())

    def createAggregations(self, popDf, litDf, empDf):
        areaCol = "Total__Rural__Urban" if "Total__Rural__Urban" in popDf.columns else "Total_Rural_Urban"

        # 1. Pivoted Gender Table
        popDf.groupBy("State_Name", "Gender").pivot("Year").agg(F.sum("Value")) \
            .withColumnRenamed("2011", "Total_Pop_2011").withColumnRenamed("2021", "Total_Pop_2021") \
            .write.format("delta").mode("overwrite").saveAsTable(f"{self.schemaPath}.viz_pivot_gender")

        # 2. Combined Rural/Urban Table
        pSum = popDf.groupBy("State_Name", areaCol, "Year").agg(F.sum("Value").alias("Pop"))
        lSum = litDf.filter(F.col("Literacy_Status") == "Literate").groupBy("State_Name", areaCol, "Year").agg(F.sum("Value").alias("Lit"))
        eSum = empDf.filter(F.col("Employment_Status") == "Employed").groupBy("State_Name", areaCol, "Year").agg(F.sum("Value").alias("Emp"))

        combined = pSum.join(lSum, ["State_Name", areaCol, "Year"], "left").join(eSum, ["State_Name", areaCol, "Year"], "left")

        combined.groupBy("State_Name", areaCol).pivot("Year") \
            .agg(F.first("Pop").alias("Pop"), F.first("Lit").alias("Lit"), F.first("Emp").alias("Emp")) \
            .write.format("delta").mode("overwrite").saveAsTable(f"{self.schemaPath}.viz_rural_urban_comparison")
            
        print("\n[SUCCESS] Aggregations created.")

# 2. EXECUTION & PREVIEWS
try:
    silverManager = SilverTransformer(spark, targetSchema)
    masterRegion = silverManager.fetchTable("region")
    
    # Process Base Tables
    processedTables = {}
    for name in ["population", "literacy", "employment"]:
        cleaned = silverManager.cleanAndStandardize(silverManager.fetchTable(name), masterRegion, name)
        processedTables[name] = cleaned
        cleaned.write.format("delta").mode("overwrite").saveAsTable(f"{targetSchema}.silver_{name}_final")

    # Run Aggregations
    silverManager.createAggregations(processedTables["population"], processedTables["literacy"], processedTables["employment"])

    # --- Previews ---
    print("\n" + "="*40)
    print("PREVIEW: VIZ_PIVOT_GENDER (2011 vs 2021)")
    print("="*40)
    spark.read.table(f"{targetSchema}.viz_pivot_gender").show(5)

    print("\n" + "="*40)
    print("PREVIEW: RURAL_URBAN_COMPARISON")
    print("="*40)
    spark.read.table(f"{targetSchema}.viz_rural_urban_comparison").show(5)

except Exception as err:
    print(f"Error: {err}")


def cleanup_storage():
    tables = ["viz_pivot_gender", "viz_rural_urban_comparison", "silver_population_final", "silver_literacy_final", "silver_employment_final"]
    for t in tables:
        spark.sql(f"DROP TABLE IF EXISTS {targetSchema}.{t}")
    print("Cleanup complete.")
 


[SUCCESS] Aggregations created.

PREVIEW: VIZ_PIVOT_GENDER (2011 vs 2021)
+-----------------+------+--------------+--------------+
|       State_Name|Gender|Total_Pop_2011|Total_Pop_2021|
+-----------------+------+--------------+--------------+
|       Tamil Nadu|  Male|      95909054|      89552506|
|Arunachal Pradesh|  Male|      27794976|      34568330|
|            Assam|  Male|      65626764|      58192392|
|       Tamil Nadu| Other|      95953332|      87415562|
|   Andhra Pradesh| Other|      31476854|      22610902|
+-----------------+------+--------------+--------------+
only showing top 5 rows

PREVIEW: RURAL_URBAN_COMPARISON
+-----------------+-------------------+---------+--------+--------+---------+--------+--------+
|       State_Name|Total__Rural__Urban| 2011_Pop|2011_Lit|2011_Emp| 2021_Pop|2021_Lit|2021_Emp|
+-----------------+-------------------+---------+--------+--------+---------+--------+--------+
|   Andhra Pradesh|              Urban| 18583314|14055438| 6009708|